<a href="https://colab.research.google.com/github/Hanzet22/GAN-Furry-Art-Photo/blob/main/GAN%20Furry%20V4.1%20(Art%20And%20Photo%20(Maintenance)).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HyperGaruda Real-ESRGAN — Fixed Version

**Model tersedia:**
- `anime` — Ilustrasi 2D / Furry Art
- `photo` — Foto Fursuit / Real Photo
- `sharpen-effect` — Foto Fursuit Yang Dipertajam dan Diperkuat strukturnya (Pengurangan noise kurang)
- `NKMD` — Foto Fursuit / Real, mempertajam edges dan menghapus efek artefak
- `ultrasharp` — Foto Fursuit / Real, menghapus artefak, upscale hingga 4X (coming soon X8)
- `foolhardy-remacri` — Foto Fursuit / Real, peningkatan detail ekstrem hingga pori-pori fur

Pilih model di cell berikutnya sebelum run.

Model HuggingFace sementara dalam Maintenance

---

# HyperGaruda Real-ESRGAN — Fixed Version

**Available models:**
- `anime` — 2D illustrations / Furry art
- `photo` — Fursuit photos / Real photos
- `sharpen-effect` — A fursuit photo that has been sharpened and has its structure enhanced (Less noise reduction)
- `NKMD` — Fursuit / Real photos, sharpens edges and removes compression artifacts
- `ultrasharp` — Fursuit / Real photos, removes artifacts, upscale up to 4X (X8 coming soon)
- `foolhardy-remacri` — Fursuit / Real photos, extreme detail enhancement up to fur pores

Select a model in the next cell before running.

HuggingFace model is currently under maintenance

In [ ]:
MODE = 'sharpen-effect'  # ganti ke 'photo' kalau mau upscale foto fursuit

# ==========================================
# CONFIG — PILIH MODEL DI SINI
# ==========================================
# 'anime' = ilustrasi 2D, furry art, digital art
# 'photo' = foto fursuit, foto real, IRL photo
# 'sharpen-effect' = foto fursuit, foto real, sharpen dan struktur diperkuat, Pengurangan Noise kurang
# 'NKMD' = foto fursuit, real, sharpen dan struktur diperuat, mempertajam edges, penghapusan efek artefak
# 'ultrasharp' = foto fursuit, real, sharpen dan struktur diperkuat, menghapus artefak, upscale hingga 4X (coming soon X8)
# 'foolhardy-remacri' = foto fursuit, real, peningkatan detail hingga pori pori fur

# --- GROUP 1: Real-ESRGAN Models ---
if MODE == 'anime':
    MODEL_NAME = 'RealESRGAN_x4plus_anime_6B'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus_anime_6B.pth'
elif MODE == 'photo':
    MODEL_NAME = 'RealESRGAN_x4plus'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus.pth'
elif MODE == 'sharpen-effect':
    MODEL_NAME = 'RealESRNet_x4plus'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth'
    MODEL_FILE = 'weights/RealESRNet_x4plus.pth'

# --- GROUP 2: HuggingFace Models ---
elif MODE == 'NKMD':
    MODEL_NAME = 'NKMD'
    MODEL_URL  = 'https://huggingface.co/art0123/Models_collection/resolve/main/upscale_models/4x_NMKD-Superscale-SP_178000_G.pth'
    MODEL_FILE = 'weights/4x_NMKD-Superscale-SP_178000_G.pth'
elif MODE == 'ultrasharp':
    MODEL_NAME = 'Ultrasharp'
    MODEL_URL  = 'https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth'
    MODEL_FILE = 'weights/4x-Ultrasharp.pth'
elif MODE == 'foolhardy-remacri':
    MODEL_NAME = 'Foolhardy Remacri'
    MODEL_URL  = 'https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_Foolhardy_Remacri.pth'
    MODEL_FILE = 'weights/4x_Foolhardy_Remacri.pth'
else:
    raise ValueError(f'MODE "{MODE}" tidak valid. Pilih dari model yang tersedia di atas.')

print(f'✅ Mode: {MODE.upper()} | Model: {MODEL_NAME}')

In [ ]:
import os
import site

# ==========================================
# 1. CLONE REPO & DIRECTORY SETUP
# ==========================================
print('📥 Checking Real-ESRGAN Repository...')
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git

# Always ensure we are inside the directory for the following steps
if 'Real-ESRGAN' in os.getcwd():
    print('✅ Already in Real-ESRGAN folder.')
else:
    %cd Real-ESRGAN
    print('✅ Entered Real-ESRGAN folder.')

# ==========================================
# 2. INSTALL DEPENDENCIES
# ==========================================
print('🔧 Installing dependencies (this may take a minute)...')
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!python setup.py develop -q
print('✅ Dependencies installed.')

# ==========================================
# 3. AUTOMATIC MODEL DOWNLOAD (DYNAMIC)
# ==========================================
# Ensure weights folder exists inside Real-ESRGAN
os.makedirs('weights', exist_ok=True)

# The MODEL_FILE and MODEL_URL come from the selection cell
if not os.path.exists(MODEL_FILE):
    print(f"📥 Downloading model: {MODEL_NAME}...")
    !wget -q -O {MODEL_FILE} {MODEL_URL}
    print("✅ Download complete!")
else:
    print(f"📦 Model {MODEL_NAME} is already present.")

print(f'🚀 Active Mode: {MODE.upper()}')

# ==========================================
# 4. FIX BASICSR COMPATIBILITY
# ==========================================
print('🔧 Patching basicsr compatibility...')
try:
    import basicsr
    basicsr_path = os.path.dirname(basicsr.__file__)
    files_to_patch = [
        os.path.join(basicsr_path, 'data/degradations.py'),
        os.path.join(basicsr_path, 'utils/img_process_util.py'),
    ]

    for fpath in files_to_patch:
        if os.path.exists(fpath):
            with open(fpath, 'r') as f:
                content = f.read()
            patched = content.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            with open(fpath, 'w') as f:
                f.write(patched)
            print(f'✅ Patched: {os.path.basename(fpath)}')
except Exception as e:
    print(f'⚠️ Patching failed: {e}')

print('✨ Setup Finished!')

In [ ]:
!pip uninstall -y basicsr
!pip install basicsr-fixed

In [ ]:
# Menghapus semua file di dalam folder inputs tanpa menghapus foldernya
!rm -rf inputs/*

In [ ]:
# ==========================================
# 5. UPLOAD GAMBAR
# ==========================================
from google.colab import files
import glob

os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'📂 Upload gambar ({MODE} mode)...')
uploaded = files.upload()

for filename in uploaded.keys():
    new_name = filename.replace(' ', '_')
    !mv "{filename}" "inputs/{new_name}"
    print(f'   ➕ {new_name}')

In [ ]:
# ==========================================
# 6. UPSCALE
# ==========================================
print(f'🚀 Upscaling ({MODE.upper()} mode | {MODEL_NAME})...')

!python inference_realesrgan.py \
    --model_path {MODEL_FILE} \
    --model_name {MODEL_NAME} \
    --input inputs \
    --output results \
    -s 4 \
    --ext png \
    --tile 512 \
    --tile_pad 32 \
    --fp32

In [ ]:
# ==========================================
# 7. DOWNLOAD HASIL
# ==========================================
from google.colab import files
import glob

print('\n🎉 Done! Preparing download...')

if os.path.exists('results') and os.listdir('results'):
    result_files = glob.glob('results/*.png')
    print(f'✅ {len(result_files)} image(s) upscaled.')
    !zip -r /content/results.zip results
    files.download('/content/results.zip')
    print('📥 Download started.')
else:
    print('⚠️ No results found. Check error logs above.')